# 03 · Modeling

Every guess and model is tested season by season with `src/evaluation.py`: for each of the last 9 seasons (2000–2009, no 2004), it may only learn from the seasons before, then predicts that season. The score is the average over the 9 seasons.

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd

from src.evaluation import test_seasons, evaluate_by_season, division_rank, mean_rank_error, pick_with_practice_test

team_table = pd.read_csv("../data/team_table.csv")
seasons = test_seasons(team_table)
seasons

## Task (a): ranking

Score: on average, how many positions off the predicted rank is (0 = perfect).

### Simple guess: same order as last season

Inside each division, the team with the most points per game last season is predicted 1st, the next 2nd, and so on. New teams go last. This is "same position as last season" when divisions stay the same, and still works when they change (e.g. 1998, when the league went from 4 to 6 divisions).

In [ ]:
def same_order_as_last_season(train, test):
    return division_rank(test, test["last_pts_vs_avg"])

simple_a = evaluate_by_season(team_table, same_order_as_last_season, mean_rank_error, seasons)
print("Average positions off:", round(simple_a["score"].mean(), 2))
simple_a

### For comparison: random order

What a guess with no information scores. Any model has to be clearly better than this, and better than the simple guess above.

In [ ]:
rng = np.random.default_rng(0)

def random_order(train, test):
    return division_rank(test, rng.random(len(test)))

random_a = evaluate_by_season(team_table, random_order, mean_rank_error, seasons)
print("Average positions off:", round(random_a["score"].mean(), 2))

### First models (first try)

Each model learns, from the training seasons only, a team's **relative position** in its division (`rank / div_size`, so 1st of 5 and 1st of 7 are comparable). Its prediction is then turned into positions 1, 2, 3… inside each division with `division_rank`.

- **Linear regression, 3 features:** last season's points and the two roster columns.
- **Linear regression, all features.**
- **Random forest, all features:** many decision trees averaged; can pick up non-linear patterns.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

ALL_FEATURES = ["last_pts_vs_avg", "last_gf_vs_avg", "last_ga_vs_avg", "last_made_playoffs", "last_rank",
                "new_team", "roster_pts_vs_avg", "roster_goalie_ga_vs_avg", "div_size"]
FEW_FEATURES = ["last_pts_vs_avg", "roster_pts_vs_avg", "roster_goalie_ga_vs_avg"]

def model_predict(make_model, features):
    """A predict(train, test) function for the test loop: learn relative position, return ranks."""
    def predict(train, test):
        model = make_model().fit(train[features], train["rank"] / train["div_size"])
        return division_rank(test, -model.predict(test[features]))   # lower predicted position = better
    return predict

models = {
    "Linear, 3 features": model_predict(LinearRegression, FEW_FEATURES),
    "Linear, all features": model_predict(LinearRegression, ALL_FEATURES),
    "Random forest, all features": model_predict(
        lambda: RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=0), ALL_FEATURES),
}
results = {"Random order": random_a, "Simple guess": simple_a}
for name, predict in models.items():
    results[name] = evaluate_by_season(team_table, predict, mean_rank_error, seasons)

summary = pd.DataFrame({name: r.set_index("season")["score"] for name, r in results.items()})
print(summary.mean().round(2).sort_values())
summary.round(2)

### Fair choice of settings (practice test)

In the first try we picked the best version **after** seeing the test scores, which makes it look slightly better than it is. Here each version is chosen **inside the training seasons only**: the last 3 training seasons are a practice test, the version with the best practice score is chosen, and only then is it scored on the real test season.

- **Linear:** choose between 3 features and all features.
- **Random forest:** choose how detailed the trees are: at least 5, 20 or 40 teams per leaf, and at most 3, 6 or unlimited questions in a row. Less detailed trees cannot memorise the training seasons.

This cell takes about a minute.

In [ ]:
forest_candidates = {
    f"leaf {leaf}, depth {depth}": model_predict(
        lambda leaf=leaf, depth=depth: RandomForestRegressor(n_estimators=200, min_samples_leaf=leaf, max_depth=depth,
                                                             random_state=0, n_jobs=-1), ALL_FEATURES)
    for leaf in [5, 20, 40] for depth in [3, 6, None]
}
linear_candidates = {
    "3 features": model_predict(LinearRegression, FEW_FEATURES),
    "all features": model_predict(LinearRegression, ALL_FEATURES),
}

chosen = {}   # which version was chosen for each test season

def choose_then_predict(candidates, label):
    def predict(train, test):
        best, _ = pick_with_practice_test(train, candidates, mean_rank_error)
        chosen.setdefault(label, []).append(best)
        return candidates[best](train, test)
    return predict

fair = {
    "Simple guess": simple_a,
    "Linear, chosen fairly": evaluate_by_season(team_table, choose_then_predict(linear_candidates, "Linear"), mean_rank_error, seasons),
    "Forest, chosen fairly": evaluate_by_season(team_table, choose_then_predict(forest_candidates, "Forest"), mean_rank_error, seasons),
}
fair_summary = pd.DataFrame({name: r.set_index("season")["score"] for name, r in fair.items()})
print(fair_summary.mean().round(2))
print()
print(pd.DataFrame(chosen, index=seasons))
fair_summary.round(2)

### Checks: is the difference real, and does a model memorise?

- **Better / equal / worse:** in how many of the 9 seasons the model beat, matched or lost to the simple guess.
- **Wilcoxon p-value:** how often a difference this big would appear by chance if the model were no better. Below 0.05 counts as real.
- **Training vs test error:** if the error on the seasons a model learned from is much lower than on new seasons, it memorised them (overfitting).

In [ ]:
from scipy.stats import wilcoxon

def training_error(predict):
    """Average error of a model on the seasons it learned from (same 9 refits as the test loop)."""
    return np.mean([mean_rank_error(team_table[team_table["year"] < s],
                                    predict(team_table[team_table["year"] < s], team_table[team_table["year"] < s]))
                    for s in seasons])

checks = {}
for name, r in fair.items():
    if name == "Simple guess":
        continue
    diff = r["score"] - simple_a["score"]
    checks[name] = {"better": int((diff < 0).sum()), "equal": int((diff == 0).sum()), "worse": int((diff > 0).sum()),
                    "Wilcoxon p": round(wilcoxon(r["score"], simple_a["score"]).pvalue, 2),
                    "test error": round(r["score"].mean(), 2)}

# Training error: the first-try forest vs the less detailed forest chosen most often vs linear
checks["Linear, chosen fairly"]["training error"] = round(training_error(linear_candidates["3 features"]), 2)
checks["Forest, chosen fairly"]["training error"] = round(training_error(forest_candidates["leaf 20, depth 3"]), 2)
first_try_forest = model_predict(lambda: RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=0), ALL_FEATURES)
print("First-try forest (leaf 5, unlimited depth): training error", round(training_error(first_try_forest), 2),
      "vs test error", round(results["Random forest, all features"]["score"].mean(), 2))
pd.DataFrame(checks).T

### What this shows

- The linear model with 3 features is the best (1.02 positions off vs 1.07 for the simple guess), and the practice test chooses it in every season, so the number is honest.
- No model is proven better than the simple guess (Wilcoxon p above 0.05): last season already says most of what can be known before the season, and 9 test seasons are too few to prove a small gap.
- The first-try forest memorised the training seasons (overfitting: 0.57 vs 1.07). Less detailed trees stop that, but still do not beat the simple guess.

Plain-language explanation: `docs/modeling_explained.md`.

## Task (b): coach change

Same loop: `evaluate_by_season(table, predict, score, seasons)`.

- `predict(train, test)` returns 1 (new coach) or 0 for each row of `test`, learning only from `train`.
- `score(test, predictions)` returns one number, e.g. the share of real coach changes caught (recall).
- To choose between versions fairly, use `pick_with_practice_test` (see task (a) above).